# Develop and test a quantum error-correction scheme with `qdk.ec`

A code definition, a circuit, and the meaning of its measurements are different parts of a design. Changing one can break the others. This walkthrough makes those changes deliberately: compare two implementations, repair readout equations, and find an undetected logical fault in a syndrome-extraction circuit.

`qodec` holds the declarations and saves them. `qdk.ec` computes and checks their consequences:

| Workflow | API | Question |
| --- | --- | --- |
| Analyze | `ec.SubsystemCode`, `ec.GadgetProfile` | What does this code or circuit do, and how many allowed faults can cause a logical failure? |
| Audit | `ec.audit` | Which declarations disagree or leave something unspecified? |
| Derive | `ec.derive` | Which checks and observable bindings can exact simulation supply? |
| Synthesize | `ec.build_qodec` | Which gadget circuits can we construct from a code definition? |

## Installing

`qdk.ec` is an optional extra of the `qdk` package:

```bash
pip install "qdk[ec]"
```

## 1. Load a qodec

Start from `c4.qodec.yaml`, next to this notebook. It describes the $[[4,2,2]]$ code: four physical qubits encode two logical qubits, with distance two. With ideal syndrome measurements, it detects arbitrary single-qubit data errors. Whether a circuit fault is detected is a separate question, explored below.

In [1]:
from collections import Counter
from pathlib import Path

from IPython.display import Markdown, display
import qodec as qc
import qdk.ec as ec

protocol = qc.Qodec.load("c4.qodec.yaml")
print(protocol.description)

Protocol based on Knill's C4


A qodec is a chain of **layers**, from the most abstract instruction set down to
the most concrete. Each layer carries the **gadgets** that lower one of its
instructions into a circuit over the layer below. Here there is a single lowering
edge: the logical `C4` instruction set down to physical `stim` operations.

In [2]:
layer = protocol.layers[0]
print("lowering:", layer.instruction_set.name, "->", protocol.layers[1].instruction_set.name)
print("gadgets: ", sorted(layer.gadgets))

lowering: C4 -> stim
gadgets:  ['idle', 'measure_xx', 'measure_zz', 'prepare_xx', 'prepare_zz', 'transversal_cx', 'x0', 'x1', 'z0', 'z1']


## 2. Profile the code and gadgets

`SubsystemCode` adds algebraic analysis to qodec's code data. `GadgetProfile` reports facts obtained through exact simulation.

In [3]:
code = ec.SubsystemCode.of(protocol.codes["C4"])

print("stabilizers:", list(code.stabilizers))
print("logical basis:", list(code.logical_basis))

distance, witness = code.distance()
logical_error = ec.Pauli.identity()
for factor in witness:
    logical_error *= factor

print("distance:", distance)
print("witness factors:", [str(factor) for factor in witness])
print("combined error:", logical_error)
print("weight:", logical_error.weight)
print("syndrome:", sorted(code.syndrome_of(logical_error)))
print("logical effect:", code.logical_effect_of(logical_error))

assert distance == logical_error.weight == 2
assert not code.syndrome_of(logical_error)
assert code.logical_effect_of(logical_error).weight > 0

stabilizers: [XXXX, ZZZZ]
logical basis: [XX, ZIZ, XIX, ZZ]
distance: 2
witness factors: ['X', 'IX']
combined error: XX
weight: 2
syndrome: []
logical effect: X


The returned witness is a list of Pauli factors. Their product has weight two, an empty syndrome, and a nontrivial logical effect: it changes encoded information without violating a stabilizer. That is a concrete distance-two witness.

The code cannot guarantee correction of every arbitrary single-qubit error. This does not rule out correction with extra information, nor does code distance alone certify a measurement circuit.

### Declared vs. realized action

A gadget's implemented instruction states what it should do; its circuit is an independent implementation of that instruction. `GadgetProfile.objective` describes the instruction, while `GadgetProfile.action` describes the circuit. Comparing them can catch a transcription error without treating the circuit itself as the specification.

In [4]:
measure_zz = layer.gadgets["measure_zz"]
profile = ec.GadgetProfile(measure_zz)
objective = profile.objective
assert objective is not None

print("objective:", objective)
print("action:   ", profile.action)
print("mismatch: ", profile.action.why_not_equivalent_to(objective) or "none")

objective: observables: FrameGroup(generators=(Z^{4}, IZ^{5}))
stabilizers: FrameGroup(generators=())
mapping: {}
action:    observables: FrameGroup(generators=(Z^{9,10,12}, IZ^{9,10,11}))
stabilizers: FrameGroup(generators=())
mapping: {}
mismatch:  none


### Checks and readouts

A circuit emits measurement bits. Checks combine bits and encoding signs into parities that should be zero without faults. Readouts specify how those measurements give the instruction's logical results or flags.

The profile below returns **measurement-record projections**: integers index `measure_zz.circuit.readouts`. These projections omit boundary-frame terms, so they are not complete qodec reference equations. Inspect the gadget's declared readouts alongside them.

Exact simulation can derive checks and observable bindings for supported circuits. Flag equations must be supplied by the author, and an audit remains necessary.

In [5]:
print("Discovered checks, measurement positions only:")
for positions in profile.checks:
    print(sorted(positions))

display(Markdown("\n".join([
    "| Readout | Profile measurement positions | Authored equation |",
    "| ---: | --- | --- |",
    *(f"| {position} | `{sorted(positions)}` | `{readout}` |"
      for position, (positions, readout) in enumerate(zip(profile.readouts, measure_zz.readouts))),
])))

Discovered checks, measurement positions only:
[0, 1, 2, 3]


| Readout | Profile measurement positions | Authored equation |
| ---: | --- | --- |
| 0 | `[0, 2]` | `["circuit.readouts[0]", "circuit.readouts[2]"]` |
| 1 | `[0, 1]` | `["circuit.readouts[0]", "circuit.readouts[1]"]` |

## 3. Audit and repair the readout equations

The circuit can implement the right logical measurement while its declared readout equation reports the wrong answer. The authored `measure_xx` and `measure_zz` readouts omit the incoming logical-frame signs: their parities work for a zero frame, but not for an arbitrary known Pauli correction.

`ec.audit` returns errors, warnings, and informational findings without changing the protocol. First summarize them by rule and inspect one error. Its filename and one-based line locate the loaded declaration; the layer and equation indices are zero-based.

In [6]:
report = ec.audit(protocol)
counts = Counter((item.severity.name, item.rule) for item in report.diagnostics)
display(Markdown("\n".join([
    "| Severity | Rule | Count |",
    "| --- | --- | ---: |",
    *(f"| {severity} | `{rule}` | {count} |"
      for (severity, rule), count in sorted(counts.items())),
])))

readout_error = next(item for item in report.errors if item.rule == "gadget/readout-mismatch")
print(ec.Report((readout_error,)))

readout_node = protocol.resolve('layers[0].gadgets["measure_xx"].readouts[0].equation')
assert readout_node.source_location is not None
assert readout_error.source_location is not None
assert readout_node.source_location.path == readout_error.source_location.path
assert readout_node.source_location.line == readout_error.source_location.line

| Severity | Rule | Count |
| --- | --- | ---: |
| ERROR | `gadget/readout-mismatch` | 4 |
| WARNING | `gadget/incomplete-output-frame` | 12 |

[ERROR] gadget/readout-mismatch
~/repositories/qdk/samples/notebooks/qdk_ec/c4.qodec.yaml:241
layers[0].gadgets['measure_xx'] (C4 -> stim)
readouts[0] does not report the required logical X_0 measurement
    Declared equation: ["circuit.readouts[0]", "circuit.readouts[1]"]
    Verified readout equation: ["in[0].x[0]", "circuit.readouts[0]", "circuit.readouts[1]"]

audit: 1 error(s), 0 warning(s), 0 informational


### Apply the missing frame signs

For each logical X measurement, include the corresponding `in[0].x[...]` sign; for logical Z, include `in[0].z[...]`. The verified equation above shows the first correction. Apply the same reasoning to both outputs of each measurement gadget on a fresh copy, without parsing a diagnostic's prose into code.

In [7]:
repaired = qc.Qodec.load("c4.qodec.yaml")
for mnemonic, basis in (("measure_xx", "x"), ("measure_zz", "z")):
    measurement = repaired.layers[0].gadgets[mnemonic]
    measurement.readouts = [
        [*readout.equation, f"in[0].{basis}[{position}]"]
        for position, readout in enumerate(measurement.readouts)
    ]

repaired_report = ec.audit(repaired)
display(Markdown("\n".join([
    "| Version | Errors | Warnings | INFO | report.ok |",
    "| --- | ---: | ---: | ---: | --- |",
    *(f"| {name} | {len(result.errors)} | {len(result.warnings)} | "
      f"{len(result.informational)} | {result.ok} |"
      for name, result in (("Authored", report), ("Frame signs added", repaired_report))),
])))

assert repaired_report.ok
assert len(repaired_report.warnings) == 12
assert len(ec.audit(protocol).errors) == 4

| Version | Errors | Warnings | INFO | report.ok |
| --- | ---: | ---: | ---: | --- |
| Authored | 4 | 12 | 0 | False |
| Frame signs added | 0 | 12 | 0 | True |

The four readout errors are gone. The twelve warnings still identify output stabilizer signs not determined by the declared relations. `report.ok` means **no errors**, not that every concern is resolved or that the protocol is fault tolerant.

## 4. Derive missing declarations

Now remove checks from a separate gadget draft and ask `ec.derive` to supply them. Keep its authored readouts so the experiment isolates missing checks. Derivation returns a new artifact; it is not a replacement for audit, and it does not infer flag equations.

In [8]:
draft = qc.Gadget(
    measure_zz.implements,
    measure_zz.circuit,
    inputs=measure_zz.inputs,
    outputs=measure_zz.outputs,
    checks=[],
    readouts=measure_zz.readouts,
    parameter_bindings=measure_zz.parameter_bindings,
    metadata=measure_zz.metadata,
)
completed = ec.derive(draft)
assert isinstance(completed, qc.Gadget)
assert not draft.checks and completed.checks

display(Markdown("\n".join([
    "| Version | Declared checks | Declared readouts |",
    "| --- | ---: | ---: |",
    *(f"| {name} | {len(item.checks)} | {len(item.readouts)} |"
      for name, item in (("Draft", draft), ("Derived", completed))),
])))
for check in completed.checks:
    print([str(term) for term in check])

| Version | Declared checks | Declared readouts |
| --- | ---: | ---: |
| Draft | 0 | 2 |
| Derived | 1 | 2 |

['circuit.readouts[0]', 'circuit.readouts[1]', 'circuit.readouts[2]', 'circuit.readouts[3]', 'in[0].stabilizers[1]']


### Derivation is not automatic repair

`ec.derive` can also process every gadget in a qodec. The current C4 derivation still leaves the incoming-frame omissions and incomplete output-frame relations reported by audit. Compare counts rather than assuming a completed artifact is certified.

In [9]:
completed_protocol = ec.derive(protocol)
assert isinstance(completed_protocol, qc.Qodec)
derived_report = ec.audit(completed_protocol)

display(Markdown("\n".join([
    "| Version | Errors | Warnings | INFO |",
    "| --- | ---: | ---: | ---: |",
    *(f"| {name} | {len(result.errors)} | {len(result.warnings)} | {len(result.informational)} |"
      for name, result in (("Authored", report), ("Derived", derived_report))),
])))
assert completed_protocol is not protocol
assert Counter(item.rule for item in derived_report.diagnostics) == Counter(
    item.rule for item in report.diagnostics
)

| Version | Errors | Warnings | INFO |
| --- | ---: | ---: | ---: |
| Authored | 4 | 12 | 0 |
| Derived | 4 | 12 | 0 |

## 5. Compare implementations of the same instruction

The logical `x0` gadget uses `X 0 1`. Try `X 2 3` instead, keeping its implemented instruction and encodings fixed. The two physical operators differ by the code's X stabilizer, `X_0 X_1 X_2 X_3`, so they should act identically on encoded states.

Also try `X 0`, which is not the same logical operation. Structural equality compares declarations; `GadgetProfile.is_equivalent_to` compares their realized logical actions and boundary encodings.

In [10]:
original_x = layer.gadgets["x0"]
original_profile = ec.GadgetProfile(original_x)
comparison_rows = [
    "| Circuit | Structurally equal | Logically equivalent | Explanation |",
    "| --- | --- | --- | --- |",
]
equivalence = {}
for source in (original_x.circuit.source, "X 2 3", "X 0"):
    candidate = qc.Gadget(
        original_x.implements,
        qc.gadgets.Circuit(original_x.circuit.instruction_set, source, format="stim"),
        inputs=original_x.inputs,
        outputs=original_x.outputs,
        checks=original_x.checks,
        readouts=original_x.readouts,
        parameter_bindings=original_x.parameter_bindings,
        metadata=original_x.metadata,
    )
    candidate_profile = ec.GadgetProfile(candidate)
    equivalent = original_profile.is_equivalent_to(candidate_profile)
    equivalence[source.strip()] = equivalent
    reason = original_profile.why_not_equivalent_to(candidate_profile) or "Same logical action"
    comparison_rows.append(
        f"| `{source.strip()}` | {original_x == candidate} | {equivalent} | {reason} |"
    )
display(Markdown("\n".join(comparison_rows)))
assert equivalence == {"X 0 1": True, "X 2 3": True, "X 0": False}

| Circuit | Structurally equal | Logically equivalent | Explanation |
| --- | --- | --- | --- |
| `X 0 1` | True | True | Same logical action |
| `X 2 3` | False | True | Same logical action |
| `X 0` | False | False | Logical actions differ in their outcome-dependent Pauli signs. |

## 6. Follow faults through a gadget

Noiseless equivalence does not say how an implementation responds to faults. The `idle` gadget measures the C4 stabilizers using ancillas 4 and 5. First number its parsed calls: `FaultEvent.after` uses these zero-based call positions, not source-file line numbers.

Inject an X error on the syndrome ancilla after its first coupling, then compare X and Z errors on the data qubit at the same point. `effects_of` evaluates the whole list in one simulation. `fault_effects` provides a larger, canonical list of X and Z faults after each call on its touched qubits.

In [11]:
idle = layer.gadgets["idle"]
idle_profile = ec.GadgetProfile(idle)
calls = idle.circuit.calls
for position, call in enumerate(calls):
    print(f"{position:2d}: {call.mnemonic} {' '.join(map(str, call.operands))}")

coupling = next(
    position for position, call in enumerate(calls)
    if call.mnemonic == "CX" and call.operands == [4, 0]
)
fault_cases = [
    ("X on ancilla 4", ec.FaultEvent.after(coupling, ec.Pauli({4: "X"}))),
    ("X on data 0", ec.FaultEvent.after(coupling, ec.Pauli({0: "X"}))),
    ("Z on data 0", ec.FaultEvent.after(coupling, ec.Pauli({0: "Z"}))),
]
effects = idle_profile.effects_of([fault for _, fault in fault_cases])
fault_rows = [
    "| Fault after call | Injected error | Checks flipped | Readouts flipped | Output Pauli |",
    "| ---: | --- | --- | --- | --- |",
]
for (label, fault), effect in zip(fault_cases, effects):
    output = ", ".join(
        f"block {entry}: {error}" for entry, error in sorted(effect.output_error.items())
    ) or "none"
    fault_rows.append(
        f"| {coupling} | {label} | {sorted(effect.syndrome)} | "
        f"{sorted(effect.readout_flips)} | {output} |"
    )
display(Markdown("\n".join(fault_rows)))
assert effects[0].syndrome == {1}
assert effects[0].output_error[0].weight > 0
assert effects[2].syndrome == {2}

 0: R 4
 1: R 5
 2: H 4
 3: CX 4 0
 4: CX 4 1
 5: CX 4 2
 6: CX 4 3
 7: H 4
 8: CX 0 5
 9: CX 1 5
10: CX 2 5
11: CX 3 5
12: M 4
13: M 5


| Fault after call | Injected error | Checks flipped | Readouts flipped | Output Pauli |
| ---: | --- | --- | --- | --- |
| 3 | X on ancilla 4 | [1] | [] | block 0: XX |
| 3 | X on data 0 | [1] | [] | block 0: XX |
| 3 | Z on data 0 | [2] | [] | block 0: ZZ |

The ancilla X fault propagates through later couplings onto the data. It and the data X fault produce the same reported effects here; different causes need not have different syndromes.

Check indices refer to `idle.checks`, readout indices to `idle.readouts`, and output entries to `idle.outputs`. Complete check equations include output-sign terms: the data Z fault flips check 2. The output Pauli records flips of encoded logical probes, not the full physical error, so it does not by itself show whether a residual preserves the output codespace.

### Find an undetected logical fault

The code distance counts data errors. **Gadget distance counts circuit faults**, including their propagation through later calls. `distance()` finds the smallest allowed fault combination that leaves all declared checks unchanged, preserves each output codespace, and changes the realized logical action. This idle gadget has no logical readouts; the search uses its encoded outputs.

The default fault set contains every nonidentity Pauli on each call's qubit support, injected after the call: three possibilities for a one-qubit call and fifteen for a two-qubit call. Each allowed event costs one, even if it affects two qubits. This is different from the compact X/Z basis in `fault_effects`. A fault after a measurement does not retroactively flip its recorded bit.

`distance_bounds()` returns lower and upper bounds, with a witness for the upper bound. Equal bounds establish the exact distance. Below, replay the exact witness as one combined `FaultEvent` and inspect its reported effects.

In [12]:
gadget_distance, distance_witness = idle_profile.distance()
lower_bound, upper_bound, bounds_witness = idle_profile.distance_bounds()

print("Circuit fault distance:", gadget_distance)
print("Distance bounds:", (lower_bound, upper_bound))
assert distance_witness and bounds_witness
assert gadget_distance == len(distance_witness)
assert lower_bound <= gadget_distance <= upper_bound == len(bounds_witness)

witness_rows = [
    "| Factor | Call index | Instruction | Injected Pauli |",
    "| ---: | ---: | --- | --- |",
]
combined_fault = ec.FaultEvent({})
for factor_index, factor in enumerate(distance_witness, start=1):
    combined_fault *= factor
    for call_index, error in sorted(factor.locations.items()):
        call = calls[call_index]
        instruction = f"{call.mnemonic} {' '.join(map(str, call.operands))}"
        witness_rows.append(f"| {factor_index} | {call_index} | `{instruction}` | `{error}` |")
display(Markdown("\n".join(witness_rows)))

(witness_effect,) = idle_profile.effects_of([combined_fault])
print("Combined checks flipped:", sorted(witness_effect.syndrome))
print("Combined readouts flipped:", sorted(witness_effect.readout_flips))
print("Output logical Paulis:", dict(witness_effect.output_error))
assert not witness_effect.syndrome
assert any(error.weight for error in witness_effect.output_error.values())

Circuit fault distance: 1
Distance bounds: (1, 1)


| Factor | Call index | Instruction | Injected Pauli |
| ---: | ---: | --- | --- |
| 1 | 4 | `CX 4 1` | `IIIIX` |

Combined checks flipped: []
Combined readouts flipped: []
Output logical Paulis: {0: X}


The code distance is **two**, but this gadget's circuit fault distance is **one**. In this witness, $X_4$ after `CX 4 1` spreads to $X_2 X_3$ through the remaining couplings. `H 4` turns the ancilla's X into Z, which does not flip `M 4`. The two data X errors flip ancilla 5 twice, so its measurement is unchanged too. The data error is logical, yet all four declared checks remain zero.

The distance search also requires the combined output error to commute with the output-code stabilizers. That requirement is separate from the declared checks; individual fault factors can violate it if their output syndromes cancel together. Failure is judged against the gadget's action: for example, a logical Z on a prepared logical zero is harmless. Noiseless validity still belongs to `ec.audit`, not to the distance calculation.

### Change the allowed fault set

Supplying `faults=` replaces the default set. For comparison, allow only single-qubit X/Y/Z errors on the output data after the final call. This removes faults inside the circuit, so no ancilla error can propagate. The gadget and its declarations are unchanged; only the question has changed.

In [13]:
output_only_faults = [
    ec.FaultEvent.after(len(calls) - 1, ec.Pauli({int(label): basis}))
    for label in idle.outputs[0].support
    for basis in ("X", "Y", "Z")
]
output_distance, output_witness = idle_profile.distance(faults=output_only_faults)

display(Markdown("\n".join([
    "| Allowed faults | Minimum fault count |",
    "| --- | ---: |",
    f"| Full circuit fault set | {gadget_distance} |",
    f"| Single-qubit X/Y/Z errors after the final call | {output_distance} |",
])))

output_error = ec.Pauli.identity()
for factor in output_witness:
    output_error *= factor.locations[len(calls) - 1]
print("Output-only witness:", output_error)
assert gadget_distance == 1
assert output_distance == len(output_witness) == 2
assert code.is_non_trivial_logical_error(output_error)

| Allowed faults | Minimum fault count |
| --- | ---: |
| Full circuit fault set | 1 |
| Single-qubit X/Y/Z errors after the final call | 2 |

Output-only witness: XX


With faults confined to the output data, two are needed. That is not a repair: it simply excludes faults inside the circuit. Instead, keep the full circuit fault model and change the implementation.

### Restore the gadget's distance

Use the C4 self-checking circuit in [Ben W. Reichardt, *Fault-tolerant quantum error correction for Steane's seven-qubit color code with few or no extra qubits*, page 4, Sec. II.2](https://arxiv.org/pdf/1804.06995#page=4). The two syndrome ancillas flag dangerous faults in each other through carefully ordered couplings to the data. The circuit still uses eight CNOTs and two ancillas, with no extra check equation.

In the paper the data qubits are numbered 1-4. Here they are 0-3, with qubit 4 measuring the X stabilizer and qubit 5 measuring the Z stabilizer. We keep logical wire identities fixed and omit the diagram's shaded ancilla swaps, which accommodate its geometric layout. This example assumes the required couplings are available; swap and movement faults are not included.

Replace the `idle` circuit in `repaired`, the copy with corrected readout equations from Section 3. Keep its implemented instruction, encodings, and checks. Then compare the logical actions, re-audit the protocol, and rerun distance and bounds with the same default rule: every nonidentity Pauli after each call, at unit cost per event.

In [22]:
reichardt_source = """R 4 5
H 4
CX 4 0
CX 2 5
CX 0 5
CX 1 5
CX 4 2
CX 4 3
CX 4 1
CX 3 5
H 4
M 4 5
"""

self_checking_idle = repaired.layers[0].gadgets["idle"]
self_checking_idle.circuit = qc.gadgets.Circuit(
    idle.circuit.instruction_set, reichardt_source, format="stim"
)
reichardt_profile = ec.GadgetProfile(self_checking_idle)
assert reichardt_profile.is_equivalent_to(idle_profile)
assert self_checking_idle.checks == idle.checks
assert Counter(call.mnemonic for call in self_checking_idle.circuit.calls) == Counter(
    call.mnemonic for call in calls
)
assert self_checking_idle.circuit.source != idle.circuit.source

reichardt_distance, reichardt_witness = reichardt_profile.distance()
reichardt_lower, reichardt_upper, reichardt_bounded_witness = reichardt_profile.distance_bounds()
assert reichardt_distance == reichardt_lower == reichardt_upper == 2
assert len(reichardt_witness) == len(reichardt_bounded_witness) == 2

distance_repair_report = ec.audit(repaired)
assert distance_repair_report.ok
display(Markdown("\n".join([
    "| Idle circuit | Distance | Bounds |",
    "| --- | ---: | --- |",
    f"| Sequential syndrome extraction | {gadget_distance} | {(lower_bound, upper_bound)} |",
    f"| Reichardt self-checking order | {reichardt_distance} | {(reichardt_lower, reichardt_upper)} |",
])))
print("Revised idle circuit:")
print(self_checking_idle.circuit.source)

| Idle circuit | Distance | Bounds |
| --- | ---: | --- |
| Sequential syndrome extraction | 1 | (1, 1) |
| Reichardt self-checking order | 2 | (2, 2) |

Revised idle circuit:
R 4 5
H 4
CX 4 0
CX 2 5
CX 0 5
CX 1 5
CX 4 2
CX 4 3
CX 4 1
CX 3 5
H 4
M 4 5



### Why the new order catches the faults

An X fault on ancilla 4 after `CX 4 2` still spreads to two data qubits, but now flips the Z-syndrome measurement on ancilla 5. Conversely, a Z fault on ancilla 5 after `CX 0 5` flips the X-syndrome measurement on ancilla 4. These are the two marked fault locations in the paper's C4 circuit.

The output logical Paulis below are nontrivial, but so are the declared check syndromes: the faults are detected. The distance search covers all allowed single-call Pauli faults, not just these two examples, and finds that two faults are needed for an undetected logical failure.

In [23]:
self_checking_calls = self_checking_idle.circuit.calls
hook_cases = [
    ("X on ancilla 4", [4, 2], ec.Pauli({4: "X"})),
    ("Z on ancilla 5", [0, 5], ec.Pauli({5: "Z"})),
]
hook_faults = [
    ec.FaultEvent.after(
        next(position for position, call in enumerate(self_checking_calls)
             if call.mnemonic == "CX" and call.operands == operands),
        error,
    )
    for _, operands, error in hook_cases
]
hook_effects = reichardt_profile.effects_of(hook_faults)
display(Markdown("\n".join([
    "| Ancilla fault | After gate | Checks flipped | Output logical Pauli |",
    "| --- | --- | --- | --- |",
    *(f"| {label} | `CX {operands[0]} {operands[1]}` | {sorted(effect.syndrome)} | `{effect.output_error[0]}` |"
      for (label, operands, _), effect in zip(hook_cases, hook_effects)),
])))
assert hook_effects[0].syndrome == {1, 3}
assert hook_effects[1].syndrome == {0, 2}

| Ancilla fault | After gate | Checks flipped | Output logical Pauli |
| --- | --- | --- | --- |
| X on ancilla 4 | `CX 4 2` | [1, 3] | `IX` |
| Z on ancilla 5 | `CX 0 5` | [0, 2] | `Z` |

The self-checking gadget now matches the code's distance of two under the full circuit fault model used here. The audit reports no errors; the protocol's remaining output-frame warnings still need attention. This is a verified improvement to this gadget, not a complete fault-tolerance certificate or a model of idle, movement, or classical readout faults.

Both methods accept `solver="enumeration"`, `"mwpf"`, or `"highs"`. The examples use the defaults: enumeration for exact distance and MWPF for bounds. To use HiGHS, install `qdk[ec,ec-highs]` and pass `solver="highs"`. The fault model stays the same when the solver changes.

An exact call raises if a search cutoff or solver limit leaves the optimum unproved. A bounds call may return a gap; if its witness is empty, its numeric upper value is a sentinel, not a finite distance. Always inspect the witness as well as the numbers.

## 7. Synthesize a protocol from the code

The same C4 definition can seed a new two-layer protocol. `ec.build_qodec` constructs gadget circuits and their declarations; it does not copy the hand-authored implementation or the Reichardt circuit just installed. Inspect its instruction menu and one generated circuit, then run the same audit rather than assuming construction settled every requirement.

In [14]:
synthesized = ec.build_qodec(protocol.codes["C4"])
synthesized_gadgets = synthesized.layers[0].gadgets
display(Markdown("\n".join([
    "| Generated instruction | Circuit calls | Checks | Readouts |",
    "| --- | ---: | ---: | ---: |",
    *(f"| `{mnemonic}` | {len(gadget.circuit.calls)} | {len(gadget.checks)} | {len(gadget.readouts)} |"
      for mnemonic, gadget in sorted(synthesized_gadgets.items())),
])))
print("Generated idle circuit:")
print(synthesized_gadgets["idle"].circuit.source)
assert synthesized is not protocol
assert set(synthesized.layers[0].instruction_set.instructions) == set(synthesized_gadgets)

| Generated instruction | Circuit calls | Checks | Readouts |
| --- | ---: | ---: | ---: |
| `idle` | 16 | 4 | 0 |
| `measure_x` | 8 | 1 | 2 |
| `measure_z` | 4 | 1 | 2 |
| `prepare_x` | 24 | 3 | 0 |
| `prepare_z` | 20 | 3 | 0 |
| `x0` | 2 | 0 | 0 |
| `x1` | 2 | 0 | 0 |
| `z0` | 2 | 0 | 0 |
| `z1` | 2 | 0 | 0 |

Generated idle circuit:
R 4
H 4
CX 4 0
CX 4 1
CX 4 2
CX 4 3
H 4
R 5
H 5
CZ 5 0
CZ 5 1
CZ 5 2
CZ 5 3
H 5
M 4 5



The generated menu is not the hand-authored instruction set: for example, it has no `transversal_cx` gadget. By default, synthesis raises if one of the instructions it attempts cannot be completed and verified; it does not silently omit that instruction.

Now audit the result. The current C4 synthesizer still leaves missing incoming-frame terms in measurement readouts and incomplete output-frame relations. Successful construction is not a clean audit or a fault-tolerance certificate.

In [15]:
synthesized_report = ec.audit(synthesized)
synthesis_counts = Counter((item.severity.name, item.rule) for item in synthesized_report.diagnostics)
display(Markdown("\n".join([
    "| Severity | Rule | Count |",
    "| --- | --- | ---: |",
    *(f"| {severity} | `{rule}` | {count} |"
      for (severity, rule), count in sorted(synthesis_counts.items())),
])))
assert len(synthesized_report.errors) == 4
assert len(synthesized_report.warnings) == 8
assert all(item.source_location is None for item in synthesized_report.diagnostics)

| Severity | Rule | Count |
| --- | --- | ---: |
| ERROR | `gadget/readout-mismatch` | 4 |
| WARNING | `gadget/incomplete-output-frame` | 8 |

## 8. Save the revised design

Save `repaired`, which now contains both the corrected measurement-readout equations and the Reichardt idle circuit, along with its remaining audit warnings. `Qodec.save` takes a destination directory; with `single_file=True`, the bundle is written there under `manifest_filename`. Reload that file, compare the complete declarations, and verify that the idle gadget still has distance two.

A successful round trip checks preservation, not correctness. The separate audit and distance calls check the loaded design. Source locations belong to the loaded file revision and do not participate in structural equality.

In [24]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    repaired.save(directory, single_file=True)
    saved_path = Path(directory) / repaired.manifest_filename
    reloaded = qc.Qodec.load(saved_path)
    assert reloaded == repaired
    assert ec.audit(reloaded).ok
    reloaded_distance, _ = ec.GadgetProfile(reloaded.layers[0].gadgets["idle"]).distance()
    assert reloaded_distance == reichardt_distance == 2

print("Complete protocol round-trips:", reloaded == repaired)
print("Reloaded idle distance:", reloaded_distance)

Complete protocol round-trips: True
Reloaded idle distance: 2


## What we established

The C4 code has distance two, but the original idle gadget admits a single undetected logical fault. The witness locates the problem. Restricting the fault set hides that fault; replacing the circuit with Reichardt's self-checking order fixes it under the same circuit fault model. The logical action and checks stay the same, while distance rises from one to two. Each syndrome ancilla now detects dangerous faults in the other.

Adding incoming frame signs repairs the logical measurement readouts. The saved protocol contains both those readout repairs and the new idle circuit, and the reloaded gadget still has distance two. Remaining output-frame warnings are not resolved by this change. Synthesis supplies another implementation to inspect, not a substitute for those checks.

These are different kinds of evidence. Code distance does not certify a gadget's distance, and a distance result applies only to the chosen fault model. Noiseless correctness, fault propagation, audit, and persistence each need their own check.

For a next experiment, swap two couplings in a copy of the self-checking circuit, audit it, compare its action, and rerun `distance()` with the full circuit fault model. Inspect any new witness rather than comparing only the reported number.